<a href="https://colab.research.google.com/github/thienbao21022k7-droid/Cuoiky/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
import os

# 2. CẤU HÌNH THÔNG SỐ
data_dir = '/content/drive/MyDrive/BUY_CAKE'
save_path = '/content/drive/MyDrive/cake_classifier.pth' # Nơi lưu mô hình

batch_size = 16
num_epochs = 10
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Đang chạy trên thiết bị: {device}")

# Kiểm tra thư mục có tồn tại không
if not os.path.exists(data_dir):
    raise Exception(f"Không tìm thấy thư mục {data_dir}. Vui lòng kiểm tra lại đường dẫn!")

# 3. CHUẨN BỊ DỮ LIỆU & TÁCH TRANSFORMS
# Transform cho tập Train (Có xoay/lật để AI học được nhiều góc độ)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Transform cho tập Val (Giữ nguyên bản, chỉ resize để thi cho chuẩn)
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Đọc thư mục để lấy danh sách loại bánh
temp_dataset = datasets.ImageFolder(data_dir)
class_names = temp_dataset.classes
num_classes = len(class_names)
print(f"\nPhát hiện {num_classes} loại bánh: {class_names}")
print("LƯU Ý: Hãy copy danh sách này dán vào biến 'class_names' ở code App để thứ tự khớp 100%.")

# Tự động chia ảnh: 80% học (train), 20% kiểm tra (val)
total_size = len(temp_dataset)
train_size = int(0.8 * total_size)

# Sinh danh sách chỉ số (indices) ngẫu nhiên
indices = torch.randperm(total_size).tolist()
train_indices = indices[:train_size]
val_indices = indices[train_size:]

# Gắn transforms tương ứng cho 2 tập
train_dataset = Subset(datasets.ImageFolder(data_dir, transform=train_transforms), train_indices)
val_dataset = Subset(datasets.ImageFolder(data_dir, transform=val_transforms), val_indices)

# KHẮC PHỤC LỖI FILENOTFOUND: Đặt num_workers=0 để tránh nghẽn Google Drive
dataloaders = {
    'train': DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0),
    'val': DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
}
dataset_sizes = {'train': len(train_dataset), 'val': len(val_dataset)}

# 4. TẠO MÔ HÌNH CNN (Dùng ResNet18)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes) # Đổi ngõ ra cho bằng số loại bánh
model = model.to(device)

criterion = nn.CrossEntropyLoss()
# TỐI ƯU: Hạ learning rate xuống 0.0001 (1e-4) để học từ tốn, không phá vỡ kiến thức cũ
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# 5. BẮT ĐẦU QUÁ TRÌNH HỌC (TRAINING)
print("\nBẮT ĐẦU HUẤN LUYỆN...")
for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}')
    print('-' * 10)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / dataset_sizes[phase]
        epoch_acc = running_corrects.double() / dataset_sizes[phase]

        print(f'{phase.upper()} | Loss: {epoch_loss:.4f} - Độ chính xác: {epoch_acc*100:.2f}%')

# 6. LƯU KẾT QUẢ
torch.save(model.state_dict(), save_path)
print(f"\n✅ HOÀN TẤT! Đã lưu 'bộ não' của AI vào: {save_path}")

Đang chạy trên thiết bị: cuda:0

Phát hiện 10 loại bánh: ['Bánh chuối nướng', 'Bánh da lợn', 'Bánh mì bơ (Cua lớn)', 'Bánh mì dừa lưới', 'Chà bông cây', 'Cookies dừa', 'Croissant', 'Egg Tart', 'MuffinViệt Quất', 'Patechaud']
LƯU Ý: Hãy copy danh sách này dán vào biến 'class_names' ở code App để thứ tự khớp 100%.

BẮT ĐẦU HUẤN LUYỆN...
Epoch 1/10
----------
TRAIN | Loss: 0.3602 - Độ chính xác: 91.35%
VAL | Loss: 0.0252 - Độ chính xác: 100.00%
Epoch 2/10
----------
TRAIN | Loss: 0.0434 - Độ chính xác: 98.99%
VAL | Loss: 0.0329 - Độ chính xác: 99.43%
Epoch 3/10
----------
TRAIN | Loss: 0.0218 - Độ chính xác: 99.86%
VAL | Loss: 0.0061 - Độ chính xác: 100.00%
Epoch 4/10
----------
TRAIN | Loss: 0.0139 - Độ chính xác: 99.71%
VAL | Loss: 0.0087 - Độ chính xác: 100.00%
Epoch 5/10
----------
TRAIN | Loss: 0.0254 - Độ chính xác: 99.28%
VAL | Loss: 0.0081 - Độ chính xác: 99.71%
Epoch 6/10
----------
TRAIN | Loss: 0.0133 - Độ chính xác: 99.64%
VAL | Loss: 0.0070 - Độ 